# 02d · Quick fix for v1: fine-tune EfficientNet-B0 on FF++ + Celeb-DF, then bake in the calibration

v1 keys on image quality: a mildly blurred real photo scores P(fake) ≈ 1. Until v2 is trained, this notebook
fine-tunes the v1 EfficientNet-B0 for a short run (about 15 min on a T4):

- **Data:** FF++ train plus the Celeb-DF crops, split **by identity** into train / val / test (60 / 15 / 25),
  because the only Celeb-DF crops are the official test list. Every frame of a person lands in one split.
- **Augmentation:** the same random blur, downscale, JPEG, noise and sharpening for real and fake images, so image
  quality stops predicting the label. No self-blended fakes (they need landmark caches).
- **Calibration:** a temperature fitted on the pooled validation set is folded into the model's last layer, and
  the checkpoint's threshold becomes the calibrated two-way one. The demo, Grad-CAM and the LLM stages read it
  like any other checkpoint.

The Celeb-DF test split here is in-domain, so a score on it isn't comparable with v1's cross-dataset Celeb-DF number.

**Notebook settings (right sidebar)**
- **Accelerator:** GPU T4 · **Internet:** On
- **Inputs:** FF++ crops (01a output, `notebookd4286b8937`), Celeb-DF crops (01b output, `celebdfv2`), v1
  checkpoints (02a output, `modelv2`)

**Produces:** `/kaggle/working/outputs/checkpoints/ft/best_calibrated.pt`. Download it to
`outputs/checkpoints/ft/best_calibrated.pt` and `python app.py` offers it in the Detector menu.


In [ ]:
# --- Project code -------------------------------------------------------------------------
# Always clones the latest code. It lives in /tmp, OUTSIDE /kaggle/working, so it is never saved into this
# notebook's output (otherwise later notebooks would pick up a stale copy). For a private repo add a
# GITHUB_TOKEN secret. Fallback: a zip of the project uploaded as a Kaggle *dataset*.
import os, sys, glob, shutil, subprocess

PROJECT = "/tmp/dfdc-project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "neo"

if not os.path.exists(PROJECT):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token
    else:
        found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
        if not found:
            raise RuntimeError("git clone failed: add a GITHUB_TOKEN secret or attach the project zip as a dataset")
        print("WARNING: git clone failed; using the attached copy at", os.path.dirname(os.path.dirname(found[0])))
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))

shutil.rmtree("/kaggle/working/project", ignore_errors=True)  # remove stale copies saved by older notebook versions
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
!git -C {PROJECT} log -1 --format="code version: %h  %s  (%cd)" --date=short 2>/dev/null || echo "code version: attached copy (no git)"

In [ ]:
from src.kaggle_utils import setup, find, find_datasets
setup(data=True)

import os
from pathlib import Path
import torch

DATA = find_datasets()
print("datasets:", DATA)
assert "CelebDF" in DATA, "Add the Celeb-DF crops (01b output) as an input"
V1 = find(os.path.join("checkpoints", "best.pt"))
assert V1, "Add the v1 checkpoints (02a output) as an input"
print("v1 checkpoint:", V1, torch.load(V1, map_location="cpu", weights_only=False).get("backbone"))


## Settings

In [ ]:
EPOCHS, SAMPLES_PER_EPOCH, LR = 3, 20000, 5e-5   # ~60k images: a short fine-tune, not a retrain
VAL_SHARE, TEST_SHARE = 0.15, 0.25               # Celeb-DF identity split
VAL_MAX_PER_SET = 3000
THRESHOLD_RULE = "youden"   # "youden": best balanced accuracy; "fpr": at most 5% of validation reals called FAKE
OUT = Path("/kaggle/working/outputs/checkpoints/ft/best.pt")


## 1 · Split the Celeb-DF crops by identity

In [ ]:
CDF = Path(DATA["CelebDF"])
if (CDF / "train").is_dir():
    print("Celeb-DF already has a train split: using it as is")
    CDF_SPLIT = CDF
else:
    CDF_SPLIT = Path("/kaggle/working/data_celebdf_split")
    !python -m src.classifier.split_crops --src "{CDF}" --out "{CDF_SPLIT}" --val {VAL_SHARE} --test {TEST_SHARE}


## 2 · Fine-tune v1

In [ ]:
args = (f'--init "{V1}" --out "{OUT}" --epochs {EPOCHS} --lr {LR} --samples-per-epoch {SAMPLES_PER_EPOCH} '
        f'--no-sbi --val-max-per-set {VAL_MAX_PER_SET} --workers 4 '
        f'--data "FF++={DATA["FF++"]}" --data "CelebDF={CDF_SPLIT}"')
!TQDM_MININTERVAL=60 python -m src.classifier.train {args}


## 3 · Bake the calibration into the checkpoint
Prints both threshold rules with their validation false-alarm and miss rates; `THRESHOLD_RULE` picks the one stored.

In [ ]:
!python -m src.classifier.calibrate --bake "{OUT}" --bake-threshold {THRESHOLD_RULE}
final = OUT.parent / "best_calibrated.pt"
c = torch.load(final, map_location="cpu", weights_only=False)
print(f"{final}: threshold {c['threshold']:.3f} (raw {c['raw_threshold']:.3f}), T = {c['temperature']:.3f}, "
      f"val AUC {c['val_auc']:.4f}")
print("Download it from the Output tab to outputs/checkpoints/ft/best_calibrated.pt")
